In [1]:
import json, pandas as pd
from pathlib import Path
R=Path('results') if Path('results').exists() else Path('experiments/E12A_static_context_expansion/results')
d=json.load(open(R/'e12a_analysis.json')); a,b=d['top5'],d['top11']
pd.set_option('display.width',200,'display.max_colwidth',60)

# E12A — Static context expansion (top-5 → up to top-11), Stage B

## 1. Research question
Can deterministic expansion of final context recover E09's retrieval/filtering failures without distractor regressions, or unacceptable token/latency/cost growth? Only variable: final K.

## 2. Why E09 motivated it
E09 static opportunity ceiling +4.0pp joint vs agent +0.67pp.

## 3. Why K=11
Smallest fixed K covering the six RETRIEVAL_FILTERING_LIMITED gold ranks (11, 9, 9, 8, 8, 9). No sweep, top-10 not tested.

**Interpretation:** this is *a deterministic fixed context expansion from top-5 to up to top-11, which for many shorter NDAs exposes most or all available chunks* (94/150 documents have <11 chunks) — not a small increase from top-5.

## 4-5. Frozen paired setup and artifact integrity

In [2]:
v=json.load(open(R/'pre_run_verification.json')); print(json.dumps(v['checks'],indent=1)); print(v['chunk_count_distribution'], v['tests_passed_before_run'])

{
 "candidate_150_of_150": true,
 "same_ids_and_order_as_e08b": true,
 "ranks_1_5_identical_to_control": true,
 "no_gold_leakage_keys": true,
 "k11_uniform_cap": true,
 "rerank_scores_non_increasing_frozen_order": true,
 "budget_gate_pass": true
}
{'7': 12, '10': 16, '8': 12, '11': 56, '5': 2, '3': 2, '4': 8, '2': 5, '6': 23, '9': 14} 338/338 (pytest tests/ -q, run immediately before)


## 6. Context-size comparison / 18. Token, cost, latency

In [3]:
print(pd.DataFrame({'top5':[a['input_tokens'][k] for k in ['mean','median','p90','max']],'top11':[b['input_tokens'][k] for k in ['mean','median','p90','max']]},index=['mean','median','p90','max']))
print(d['input_token_increase']); print('output tokens',a['output_tokens'],b['output_tokens'])
print('latency ms',a['latency_ms'],b['latency_ms'],d['latency_change']); print('cost',a['cost'],b['cost']); print(d['cost_projection'])

           top5        top11
mean    1148.82  1881.826667
median  1163.50  1985.500000
p90     1307.00  2522.000000
max     1784.00  3299.000000
{'abs': 733.0066666666667, 'pct': 63.80517980768674, 'forecast_mean': 1884.6}
output tokens {'mean': 708.7066666666667, 'median': 689.5, 'p90': 1235, 'max': 1859} {'mean': 760.5266666666666, 'median': 716.5, 'p90': 1245, 'max': 2052}
latency ms {'mean': 7411.432088869624, 'median': 6684.958771045785, 'p90': 12549.430458107963, 'max': 16792.086415924132} {'mean': 8202.406153347498, 'median': 7470.426354499068, 'p90': 14080.7896660408, 'max': 25266.676540952176} {'abs_ms': 790.974064477874, 'pct': 10.672351240534827}
cost {'total': 0.25569275, 'mean': 0.0017046183333333333, 'median': 0.00162725, 'p90': 0.002783, 'max': 0.0039265} {'total': 0.2987265, 'mean': 0.00199151, 'median': 0.0019515000000000001, 'p90': 0.0029355, 'max': 0.004741}
{'top5_per_1000': 1.7046183333333333, 'top11_per_1000': 1.9915100000000001, 'top5_per_8000': 13.63694666666666

## 7. Budget / actual spend

In [4]:
print(d['run'])

{'total_wall_seconds': 1230.7540405000327, 'n_cases': 150, 'run_id': 'e12a_190f715d', 'run_spend_usd': 0.2987265000000002, 'pre_run_ledger_spend_usd': 0.4090479499999999, 'post_run_ledger_spend_usd': 0.7077744499999996}


## 8-9. Candidate metrics and matched table

In [5]:
rows=[('Accuracy',a['accuracy'],b['accuracy']),('Macro-F1',a['macro_f1'],b['macro_f1'])]
for l in ['Entailment','Contradiction','NotMentioned']: rows.append((l+' recall',a['recall'][l],b['recall'][l]))
rows+= [('Evidence recall',a['evidence_recall'],b['evidence_recall']),('Evidence precision',a['evidence_precision'],b['evidence_precision']),('Source-valid evidence',a['source_valid_evidence_rate'],b['source_valid_evidence_rate']),('Joint',a['joint'],b['joint'])]
for l in ['Entailment','Contradiction','NotMentioned']: rows.append(('Joint '+l,a['joint_by_class'][l],b['joint_by_class'][l]))
for k in ['mean']: rows+=[('Mean input tok',a['input_tokens'][k],b['input_tokens'][k]),('Mean output tok',a['output_tokens'][k],b['output_tokens'][k]),('Mean latency ms',a['latency_ms'][k],b['latency_ms'][k])]
rows+=[('Total cost $',a['cost']['total'],b['cost']['total']),('Strict parse',a['parse']['strict'],b['parse']['strict'])]
t=pd.DataFrame(rows,columns=['metric','top5','top11']); t['delta']=t.top11-t.top5; t.round(4)

,metric,top5,top11,delta
0,Accuracy,0.7867,0.8067,0.0200
1,Macro-F1,0.7869,0.8078,0.0209
2,Entailment recall,0.8200,0.8800,0.0600
3,Contradiction recall,0.7600,0.7600,0.0000
4,NotMentioned recall,0.7800,0.7800,0.0000
5,Evidence recall,0.8600,0.9100,0.0500
6,Evidence precision,0.8350,0.8426,0.0076
7,Source-valid evidence,0.9417,0.9444,0.0027
8,Joint,0.7400,0.7800,0.0400
9,Joint Entailment,0.7400,0.8400,0.1000


In [6]:
print(b['confusion_matrix']); print(a['confusion_matrix'])

[[44, 3, 3], [12, 38, 0], [4, 7, 39]]
[[41, 3, 6], [10, 38, 2], [4, 7, 39]]


## 10-11. Classification and joint transitions

In [7]:
print('cls',d['cls_transitions']); print('joint',d['joint_transitions'])
for k in ['cls_recoveries','cls_regressions','joint_recoveries','joint_regressions']: print(k); print(pd.DataFrame(d[k]))

cls {'wrong_to_right': 8, 'right_to_wrong': 5, 'both_right': 113, 'both_wrong': 24}
joint {'wrong_to_right': 13, 'right_to_wrong': 7, 'both_right': 104, 'both_wrong': 26}
cls_recoveries
              case_id           gold           top5          top11
0  train::178::nda-20  Contradiction   NotMentioned  Contradiction
1  train::160::nda-10     Entailment   NotMentioned     Entailment
2  train::248::nda-10     Entailment  Contradiction     Entailment
3  train::353::nda-10     Entailment   NotMentioned     Entailment
4  train::379::nda-10     Entailment   NotMentioned     Entailment
5  train::518::nda-10     Entailment   NotMentioned     Entailment
6   train::279::nda-1   NotMentioned  Contradiction   NotMentioned
7   train::320::nda-1   NotMentioned  Contradiction   NotMentioned
cls_regressions
              case_id           gold           top5          top11
0  train::553::nda-20  Contradiction  Contradiction     Entailment
1  train::273::nda-10     Entailment     Entailment   NotMent

## 12-14. Six retrieval/filtering, six evidence-selection, 26 reasoning-limited

In [8]:
for n,v in d['buckets'].items(): print(n,v['n'],v['cls'],v['joint'],v['gold_overlap_top5_to_top11'])
pd.DataFrame(d['buckets']['RETRIEVAL_FILTERING_LIMITED']['rows']).drop(columns=['top5_evidence','top11_evidence'])

RETRIEVAL_FILTERING_LIMITED 6 {'unchanged_correct': 1, 'recovered': 4, 'unchanged_wrong': 1} {'recovered': 5, 'unchanged_fail': 1} [['(False, True)', 5], ['(False, False)', 1]]
EVIDENCE_SELECTION_LIMITED 6 {'unchanged_correct': 5, 'regressed': 1} {'recovered': 4, 'unchanged_fail': 2} [['(False, True)', 3], ['(True, True)', 1], ['(True, False)', 1], ['(False, False)', 1]]
MODEL_REASONING_LIMITED 26 {'unchanged_wrong': 22, 'recovered': 4} {'unchanged_fail': 22, 'recovered': 4} [['(None, None)', 11], ['(True, True)', 10], ['(False, True)', 2], ['(True, False)', 2], ['(False, False)', 1]]
DYNAMIC_INFORMATION_ACQUISITION 1 {'unchanged_wrong': 1} {'unchanged_fail': 1} [['(False, False)', 1]]


,case_id,gold,top5_label,top11_label,cls_transition,joint_transition,gold_in_top5,gold_in_top11,top5_gold_overlap,top11_gold_overlap,top11_source_valid,n_chunks_shown,gold_rank_expected
0,train::438::nda-2,Contradiction,Contradiction,Contradiction,unchanged_correct,recovered,False,True,False,True,True,11,8
1,train::160::nda-10,Entailment,NotMentioned,Entailment,recovered,recovered,False,True,False,True,True,11,11
2,train::247::nda-10,Entailment,NotMentioned,NotMentioned,unchanged_wrong,unchanged_fail,False,True,False,False,True,11,9
3,train::353::nda-10,Entailment,NotMentioned,Entailment,recovered,recovered,False,True,False,True,False,9,9
4,train::379::nda-10,Entailment,NotMentioned,Entailment,recovered,recovered,False,True,False,True,True,11,8
5,train::518::nda-10,Entailment,NotMentioned,Entailment,recovered,recovered,False,True,False,True,True,11,9


In [9]:
pd.DataFrame(d['buckets']['EVIDENCE_SELECTION_LIMITED']['rows']).drop(columns=['top5_evidence','top11_evidence'])

,case_id,gold,top5_label,top11_label,cls_transition,joint_transition,gold_in_top5,gold_in_top11,top5_gold_overlap,top11_gold_overlap,top11_source_valid,n_chunks_shown
0,train::318::nda-2,Contradiction,Contradiction,Contradiction,unchanged_correct,recovered,True,True,True,True,True,11
1,train::515::nda-2,Contradiction,Contradiction,Contradiction,unchanged_correct,recovered,True,True,False,True,True,11
2,train::273::nda-10,Entailment,Entailment,NotMentioned,regressed,unchanged_fail,True,True,True,False,True,11
3,train::317::nda-13,Entailment,Entailment,Entailment,unchanged_correct,recovered,True,True,False,True,True,11
4,train::352::nda-12,Entailment,Entailment,Entailment,unchanged_correct,recovered,True,True,False,True,True,9
5,train::500::nda-13,Entailment,Entailment,Entailment,unchanged_correct,unchanged_fail,True,True,False,False,False,11


## 15. Contradiction analysis

In [10]:
d['contradiction']

{'recall_top5': 0.76,
 'recall_top11': 0.76,
 'recoveries': ['train::178::nda-20'],
 'regressions': ['train::553::nda-20'],
 'joint_top5': 0.7,
 'joint_top11': 0.72,
 'joint_recoveries': ['train::178::nda-20',
  'train::318::nda-2',
  'train::438::nda-2',
  'train::515::nda-2'],
 'joint_regressions': ['train::88::nda-1',
  'train::161::nda-7',
  'train::553::nda-20']}

## 16. Distractor regression analysis (manual inspection; causes recorded in summary.md)

In [11]:
for r in d['regression_dossier']: print(r['case_id'],r['gold'],r['top5_label'],'->',r['top11_label'],'cls' if r['cls_regression'] else '','joint' if r['joint_regression'] else '','added chunks:',r['n_added_chunks'],'gold in added:',r['gold_span_in_added_chunks'])

train::88::nda-1 Contradiction Contradiction -> Contradiction  joint added chunks: 2 gold in added: False
train::161::nda-7 Contradiction Contradiction -> Contradiction  joint added chunks: 6 gold in added: False
train::553::nda-20 Contradiction Contradiction -> Entailment cls joint added chunks: 6 gold in added: False
train::226::nda-12 Entailment Entailment -> Entailment  joint added chunks: 0 gold in added: False
train::273::nda-10 Entailment Entailment -> NotMentioned cls  added chunks: 6 gold in added: False
train::375::nda-15 Entailment Entailment -> Contradiction cls joint added chunks: 1 gold in added: False
train::86::nda-1 NotMentioned NotMentioned -> Contradiction cls joint added chunks: 6 gold in added: None
train::297::nda-1 NotMentioned NotMentioned -> Contradiction cls joint added chunks: 6 gold in added: None


## 17. Short-document saturation
Group A shows identical context in both arms (≤5 chunks), so its transitions are a direct read of run-to-run model noise.

In [12]:
pd.DataFrame(d['saturation']).T.drop(columns=['cls_transitions','joint_transitions'])

,n,mean_candidate_chunks_shown,mean_control_chunks_shown,top5_acc,top11_acc,top5_joint,top11_joint,acc_delta,joint_delta
A_<=5_chunks,17,3.411765,3.411765,0.705882,0.764706,0.705882,0.705882,0.058824,0.0
B_6-10_chunks,77,7.844156,5,0.792208,0.831169,0.779221,0.818182,0.038961,0.038961
C_>=11_chunks,56,11,5,0.803571,0.785714,0.696429,0.75,-0.017857,0.053571


## 19. Paired statistics

In [13]:
print(d['mcnemar_classification']); print(d['mcnemar_joint']); print(json.dumps(d['bootstrap'],indent=1))

{'top5_only': 5, 'top11_only': 8, 'n_discordant': 13, 'p': 0.5810546875}
{'top5_only': 7, 'top11_only': 13, 'n_discordant': 20, 'p': 0.26317596435546875}
{
 "accuracy": {
  "point": 0.020000000000000018,
  "ci95": [
   -0.026666666666666727,
   0.06666666666666665
  ]
 },
 "macro_f1": {
  "point": 0.020916664887978165,
  "ci95": [
   -0.026489332588586434,
   0.06950192353800544
  ]
 },
 "contradiction_recall": {
  "point": 0.0,
  "ci95": [
   -0.05882352941176472,
   0.05769230769230771
  ]
 },
 "joint": {
  "point": 0.040000000000000036,
  "ci95": [
   -0.019999999999999907,
   0.09999999999999998
  ]
 }
}


## 20. Decision
**B — STATIC EXPANSION SHOWS LIMITED/NEUTRAL VALUE — KEEP TOP-5.** See summary.md for the reasoning.